In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
import urllib.parse

import plotly.express as px

In [2]:
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine(config=DB_CONFIG):
    user = config["user"]
    password = urllib.parse.quote_plus(config["password"])
    host = config["host"]
    port = config["port"]
    dbname = config["dbname"]
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(conn_str)

engine = get_engine()

In [3]:
SCHEMA = "a1_fct_vision_testlog_txt_processing_history"
TABLE  = "fct_vision_testlog_txt_processing_history"

sql_query = f"""
SELECT
    station,
    remark,
    end_day,
    end_time,
    result,
    goodorbad
FROM {SCHEMA}.{TABLE}
WHERE
    station IN ('FCT1','FCT2','FCT3','FCT4')
    AND remark IN ('PD','Non-PD')
    AND result <> 'FAIL'
    AND goodorbad <> 'BadFile'
ORDER BY end_day ASC, end_time ASC
"""

df = pd.read_sql(text(sql_query), engine)
df.head()

,station,remark,end_day,end_time,result,goodorbad
0,FCT2,PD,20251001,01:25:03,PASS,GoodFile
1,FCT4,PD,20251001,01:25:26,PASS,GoodFile
2,FCT3,PD,20251001,01:25:40,PASS,GoodFile
3,FCT1,PD,20251001,01:25:40,PASS,GoodFile
4,FCT2,PD,20251001,01:25:50,PASS,GoodFile


In [4]:
# 타입 정리
df["end_day"] = pd.to_datetime(df["end_day"]).dt.date

# end_time이 문자열일 수도 있어 안전 처리
# (예: '08:34:14.93' 같은 형태 포함 가능)
df["end_time_str"] = df["end_time"].astype(str)

# end_day + end_time => timestamp
df["end_ts"] = pd.to_datetime(df["end_day"].astype(str) + " " + df["end_time_str"], errors="coerce")

# 정렬(요구사항)
df = df.sort_values(["station", "remark", "end_day", "end_ts"], ascending=True).reset_index(drop=True)

# op_ct: (현재 end_ts - 이전 end_ts) seconds
df["op_ct"] = df.groupby(["station", "remark"])["end_ts"].diff().dt.total_seconds()

# month: yyyymm
df["month"] = pd.to_datetime(df["end_day"].astype(str)).dt.strftime("%Y%m")

df[["station","remark","end_day","end_time","end_ts","op_ct","month"]].head(20)

,station,remark,end_day,end_time,end_ts,op_ct,month
0,FCT1,Non-PD,2025-10-10,00:00:23,2025-10-10 00:00:23,NaN,202510
1,FCT1,Non-PD,2025-10-10,00:00:52,2025-10-10 00:00:52,29.0,202510
2,FCT1,Non-PD,2025-10-10,00:01:22,2025-10-10 00:01:22,30.0,202510
3,FCT1,Non-PD,2025-10-10,14:47:41,2025-10-10 14:47:41,53179.0,202510
4,FCT1,Non-PD,2025-10-10,14:48:15,2025-10-10 14:48:15,34.0,202510
5,FCT1,Non-PD,2025-10-10,14:48:45,2025-10-10 14:48:45,30.0,202510
6,FCT1,Non-PD,2025-10-10,14:49:15,2025-10-10 14:49:15,30.0,202510
7,FCT1,Non-PD,2025-10-10,14:49:58,2025-10-10 14:49:58,43.0,202510
8,FCT1,Non-PD,2025-10-10,14:50:27,2025-10-10 14:50:27,29.0,202510
9,FCT1,Non-PD,2025-10-10,14:51:15,2025-10-10 14:51:15,48.0,202510


In [5]:
OUT_DIR = Path("./fct_opct_boxplot_html")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _range_str(values: pd.Series):
    """이상치 값들의 min~max를 'min~max' 문자열로. 없으면 None."""
    values = values.dropna()
    if len(values) == 0:
        return None
    vmin = float(values.min())
    vmax = float(values.max())
    return f"{vmin:.1f}~{vmax:.1f}"

def summarize_group(g: pd.DataFrame):
    """
    g: 특정 (station, remark, month) 그룹의 원본 레코드들
    반환: 통계 dict + html 파일명
    """
    # op_ct 600초 초과 제외 + NaN 제외
    s = g["op_ct"].dropna()
    s = s[s <= 600]

    sample_amount = len(s)  # ⭐ 추가

    if sample_amount == 0:
        return {
            "sample_amount": 0,  # ⭐ 추가
            "op_ct_lower_outlier": None,
            "q1": None,
            "median": None,
            "q3": None,
            "op_ct_upper_outlier": None,
            "del_out_op_ct_av": None,
            "html": None,
        }

    q1 = float(s.quantile(0.25))
    med = float(s.quantile(0.50))
    q3 = float(s.quantile(0.75))
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    lower_outliers = s[s < lower_bound]
    upper_outliers = s[s > upper_bound]

    s_wo = s[(s >= lower_bound) & (s <= upper_bound)]
    avg_wo = float(s_wo.mean()) if len(s_wo) else None

    fig = px.box(
        pd.DataFrame({"op_ct": s}),
        y="op_ct",
        points="outliers",
        title=None
    )

    station = g["station"].iloc[0]
    remark  = g["remark"].iloc[0]
    month   = g["month"].iloc[0]

    html_name = f"boxplot_{station}_{remark}_{month}.html"
    html_path = OUT_DIR / html_name
    fig.write_html(str(html_path), include_plotlyjs="cdn", full_html=True)

    return {
        "sample_amount": sample_amount,  # ⭐ 추가
        "op_ct_lower_outlier": _range_str(lower_outliers),
        "q1": round(q1, 2),
        "median": round(med, 2),
        "q3": round(q3, 2),
        "op_ct_upper_outlier": _range_str(upper_outliers),
        "del_out_op_ct_av": round(avg_wo, 2) if avg_wo is not None else None,
        "html": str(html_path),
    }

# (station, remark, month) 단위로 요약 테이블 생성
summary_rows = []
for (station, remark, month), g in df.groupby(["station","remark","month"], sort=True):
    stats = summarize_group(g)
    summary_rows.append({
        "station": station,
        "remark": remark,
        "month": month,
        **stats
    })

summary_df = pd.DataFrame(summary_rows)

# 첨부 이미지처럼 id 컬럼을 맨 앞에
summary_df.insert(0, "id", range(1, len(summary_df) + 1))

# 보기 좋게 정렬(원하시면 station/remark/month 순 그대로 유지)
summary_df = summary_df.sort_values(["month","remark","station"]).reset_index(drop=True)
summary_df["id"] = range(1, len(summary_df) + 1)

summary_df

import json
import plotly.express as px

def make_boxplot_json(op_ct_series: pd.Series) -> str:
    """
    op_ct 값 시리즈로 Plotly boxplot figure를 만들고,
    DB에 저장 가능한 JSON 문자열로 반환
    """
    fig = px.box(
        pd.DataFrame({"op_ct": op_ct_series}),
        y="op_ct",
        points="outliers",
        title=None
    )
    return fig.to_json()  # <- 이 문자열을 DB에 그대로 저장

def build_plotly_json_column(df_raw: pd.DataFrame, summary_df: pd.DataFrame) -> pd.DataFrame:
    """
    summary_df (station, remark, month) 기준으로
    원본 df_raw에서 op_ct<=600 데이터만 뽑아 figure JSON을 생성해 넣음
    """
    out = summary_df.copy()
    out["plotly_json"] = None

    for i, r in out.iterrows():
        station = r["station"]
        remark  = r["remark"]
        month   = r["month"]

        # 원본에서 해당 그룹 op_ct만 추출 (<=600, NaN 제외)
        g = df_raw[
            (df_raw["station"] == station) &
            (df_raw["remark"] == remark) &
            (df_raw["month"] == month)
        ]["op_ct"].dropna()
        g = g[g <= 600]

        if len(g) == 0:
            out.at[i, "plotly_json"] = None
        else:
            out.at[i, "plotly_json"] = make_boxplot_json(g)

    return out

# 실행
summary_df2 = build_plotly_json_column(df, summary_df)

# 기존 html 컬럼이 필요 없으면 제거(선택)
if "html" in summary_df2.columns:
    summary_df2 = summary_df2.drop(columns=["html"])

summary_df2.head(20)

,id,station,remark,month,sample_amount,op_ct_lower_outlier,q1,median,q3,op_ct_upper_outlier,del_out_op_ct_av,plotly_json
0,1,FCT1,Non-PD,202510,8288,None,30.0,30.0,33.0,38.0~583.0,30.98,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."
1,2,FCT2,Non-PD,202510,8436,None,30.0,30.0,32.0,36.0~589.0,30.59,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."
2,3,FCT3,Non-PD,202510,9017,None,30.0,31.0,32.0,36.0~593.0,31.20,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."
3,4,FCT4,Non-PD,202510,9233,None,29.0,31.0,33.0,40.0~579.0,31.31,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."
4,5,FCT1,PD,202510,14251,16.0~22.0,36.0,39.0,43.0,54.0~594.0,39.32,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."
5,6,FCT2,PD,202510,14366,22.0~22.0,36.0,38.5,42.0,52.0~598.0,39.33,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."
6,7,FCT3,PD,202510,14836,18.0~18.0,36.0,37.0,42.0,52.0~599.0,38.71,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."
7,8,FCT4,PD,202510,14827,22.0~22.0,35.0,39.0,42.0,53.0~595.0,38.46,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."
8,9,FCT1,Non-PD,202511,820,None,30.0,31.0,32.0,36.0~397.0,30.73,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."
9,10,FCT2,Non-PD,202511,737,29.0~29.0,31.0,31.0,32.0,34.0~354.0,31.00,"{""data"":[{""alignmentgroup"":""True"",""boxpoints"":..."


In [6]:
import numpy as np
import pandas as pd

base = summary_df2 if "summary_df2" in globals() else summary_df

need = {"station","remark","month","del_out_op_ct_av"}
missing = need - set(base.columns)
if missing:
    raise KeyError(f"필요 컬럼 누락: {sorted(missing)}")

# FCT1~4만
b = base[base["station"].isin(["FCT1","FCT2","FCT3","FCT4"])].copy()

# 숫자형
b["del_out_op_ct_av"] = pd.to_numeric(b["del_out_op_ct_av"], errors="coerce")

# side 매핑
side_map = {"FCT1":"left","FCT2":"left","FCT3":"right","FCT4":"right"}
b["side"] = b["station"].map(side_map)

# ---------------------------
# (1) month/remark/side 별 병렬 UPH 계산
#     UPH_side = 3600 * sum(1/CT_i)
# ---------------------------
def parallel_uph(ct_series: pd.Series) -> float:
    ct = ct_series.dropna()
    ct = ct[ct > 0]
    if len(ct) == 0:
        return np.nan
    return 3600.0 * (1.0 / ct).sum()

grp_side = (
    b.groupby(["month","remark","side"], as_index=False)
     .agg(ct_list=("del_out_op_ct_av", lambda x: list(x)))
)

grp_side["uph"] = grp_side["ct_list"].apply(lambda lst: parallel_uph(pd.Series(lst)))
grp_side["ct_eq"] = np.where(grp_side["uph"] > 0, 3600.0 / grp_side["uph"], np.nan)

# 출력 형식 반올림(소수점 2자리)
grp_side["uph"] = grp_side["uph"].round(2)
grp_side["ct_eq"] = grp_side["ct_eq"].round(2)

left_right_df = grp_side.rename(columns={"side":"station"})[
    ["station","remark","month","ct_eq","uph"]
].copy()
left_right_df["final_ct"] = np.nan  # left/right는 null

# ---------------------------
# (2) whole = left UPH + right UPH
#     final_ct = 3600 / uph_whole
# ---------------------------
whole_df = (
    left_right_df.groupby(["month","remark"], as_index=False)["uph"]
                .sum()
)

whole_df["station"] = "whole"
whole_df["ct_eq"] = np.nan
whole_df["final_ct"] = np.where(whole_df["uph"] > 0, (3600.0 / whole_df["uph"]).round(2), np.nan)

# 결합
final_df_86 = pd.concat(
    [
        left_right_df[["station","remark","month","ct_eq","uph","final_ct"]],
        whole_df[["station","remark","month","ct_eq","uph","final_ct"]],
    ],
    ignore_index=True
)

# 정렬: month -> remark -> station(left,right,whole)
station_order = pd.CategoricalDtype(["left","right","whole"], ordered=True)
final_df_86["station"] = final_df_86["station"].astype(station_order)

final_df_86 = final_df_86.sort_values(["month","remark","station"]).reset_index(drop=True)
final_df_86.insert(0, "id", range(1, len(final_df_86) + 1))

final_df_86

,id,station,remark,month,ct_eq,uph,final_ct
0,1,left,Non-PD,202510,15.39,233.89,NaN
1,2,right,Non-PD,202510,15.63,230.36,NaN
2,3,whole,Non-PD,202510,NaN,464.25,7.75
3,4,left,PD,202510,19.66,183.09,NaN
4,5,right,PD,202510,19.29,186.60,NaN
5,6,whole,PD,202510,NaN,369.69,9.74
6,7,left,Non-PD,202511,15.43,233.28,NaN
7,8,right,Non-PD,202511,15.77,228.32,NaN
8,9,whole,Non-PD,202511,NaN,461.60,7.80
9,10,left,PD,202511,20.21,178.13,NaN


In [7]:
import psycopg2
from psycopg2 import sql
import pandas as pd

# =========================
# DB 설정 (동일)
# =========================
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TARGET_SCHEMA = "e1_FCT_ct"
TBL_OPCT      = "fct_op_ct"          # summary_df2 저장
TBL_WHOLE     = "fct_whole_op_ct"    # final_df_86 저장

def ensure_schema(conn, schema_name: str):
    with conn.cursor() as cur:
        cur.execute(sql.SQL("CREATE SCHEMA IF NOT EXISTS {}").format(sql.Identifier(schema_name)))
    conn.commit()

def table_exists(conn, schema_name: str, table_name: str) -> bool:
    q = """
    SELECT EXISTS (
        SELECT 1
        FROM information_schema.tables
        WHERE table_schema = %s
          AND table_name   = %s
    )
    """
    with conn.cursor() as cur:
        cur.execute(q, (schema_name, table_name))
        return bool(cur.fetchone()[0])

def get_engine_from_db_config(config=DB_CONFIG):
    from sqlalchemy import create_engine
    import urllib.parse
    pw = urllib.parse.quote_plus(config["password"])
    conn_str = f"postgresql+psycopg2://{config['user']}:{pw}@{config['host']}:{config['port']}/{config['dbname']}"
    return create_engine(conn_str)

from pandas.api.types import CategoricalDtype

def save_df_if_table_not_exists(df: pd.DataFrame, schema_name: str, table_name: str, engine):
    """
    테이블이 이미 존재하면 PASS
    없으면 테이블 생성 후 전체 저장
    """
    if df is None or len(df) == 0:
        print(f"[SKIP] {schema_name}.{table_name}: df가 비어있습니다.")
        return

    df_to_save = df.copy()

    # month는 문자열로 고정
    if "month" in df_to_save.columns:
        df_to_save["month"] = df_to_save["month"].astype(str)

    # Categorical 타입 안전 처리 (Deprecation 대응)
    for c in df_to_save.columns:
        if isinstance(df_to_save[c].dtype, CategoricalDtype):
            df_to_save[c] = df_to_save[c].astype(str)

    with psycopg2.connect(**DB_CONFIG) as conn:
        ensure_schema(conn, schema_name)

        if table_exists(conn, schema_name, table_name):
            print(f"[PASS] {schema_name}.{table_name} 이미 존재 -> 저장 생략")
            return

    df_to_save.to_sql(
        name=table_name,
        con=engine,
        schema=schema_name,
        if_exists="fail",
        index=False,
        method="multi",
        chunksize=5000
    )
    print(f"[OK] {schema_name}.{table_name} 생성 및 저장 완료 (rows={len(df_to_save)})")

# =========================
# 실행
# =========================
engine = get_engine_from_db_config(DB_CONFIG)

# 1) summary_df2 -> e1_FCT_ct.fct_op_ct
if "summary_df2" not in globals():
    raise NameError("summary_df2 변수가 없습니다. (summary_df2 생성 셀을 먼저 실행하세요)")
save_df_if_table_not_exists(summary_df2, TARGET_SCHEMA, TBL_OPCT, engine)

# 2) final_df_86 -> e1_FCT_ct.fct_whole_op_ct
if "final_df_86" not in globals():
    raise NameError("final_df_86 변수가 없습니다. (병렬 합산 셀을 먼저 실행하세요)")
save_df_if_table_not_exists(final_df_86, TARGET_SCHEMA, TBL_WHOLE, engine)

[OK] e1_FCT_ct.fct_op_ct 생성 및 저장 완료 (rows=16)
[OK] e1_FCT_ct.fct_whole_op_ct 생성 및 저장 완료 (rows=12)
